<a href="https://colab.research.google.com/github/sudipsardar922603-dotcom/MtechProjects/blob/feature%2Ftesting/Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

training model

In [4]:
# ============================
# 1. Install Required Libraries
# ============================
!pip install -q transformers datasets setfit torch accelerate scikit-learn pandas

# ============================
# 2. Load Dataset (CSV/Excel)
# ============================
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load your dataset
df = pd.read_csv("Financial_dataset.csv")   # or pd.read_excel("financial_dataset.xlsx")

# Expected columns: question, answer, language, type
print("Sample rows:\n", df.head())

# Convert answers into categorical labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["answer"])  # numeric labels for classification

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df[["question", "label", "language", "type"]])

# Split into train/test
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# ============================
# 3. Few-Shot Training with SetFit v2
# ============================
from setfit import SetFitModel, Trainer, TrainingArguments

# Load multilingual backbone
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Training arguments
args = TrainingArguments(
    output_dir="checkpoints",
    batch_size=(16, 2),
    num_epochs=(1, 16),
    num_iterations=20,
    body_learning_rate=2e-5,
    head_learning_rate=1e-2,
    use_amp=False
)

# Trainer with column mapping
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=args,
    metric="accuracy",
    column_mapping={"question": "text", "label": "label"}
)

# Train
trainer.train()

# ============================
# 4. Global Evaluation
# ============================
y_true = test_dataset["label"]
y_pred = trainer.model(test_dataset["question"])

print("Global Accuracy:", accuracy_score(y_true, y_pred))
print("Global F1-score:", f1_score(y_true, y_pred, average="weighted"))
print("Global Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ============================
# 5. Language-Wise Evaluation
# ============================
languages = df["language"].unique()

for lang in languages:
    subset = test_dataset.filter(lambda x: x["language"] == lang)
    if len(subset) == 0:
        continue

    y_true_lang = subset["label"]
    y_pred_lang = trainer.model(subset["question"])

    print(f"\n--- {lang} ---")
    print("Accuracy:", accuracy_score(y_true_lang, y_pred_lang))
    print("F1-score:", f1_score(y_true_lang, y_pred_lang, average="weighted"))
    print("Confusion Matrix:\n", confusion_matrix(y_true_lang, y_pred_lang))

Sample rows:
                                             question  answer language  \
0  एक व्यापारी ने ₹5000 का सामान खरीदा और 18% GST...    5900    Hindi   
1  একজন ব্যবসায়ী ১০,০০০ টাকা ঋণ নিলেন ১০% বার্ষি...   11000  Bengali   
2  ஒரு வியாபாரி ₹2000 மதிப்புள்ள பொருட்களை வாங்கி...    2240    Tamil   
3  A person deposited ₹5000 in a bank at 5% annua...    5500  English   
4  एक कंपनी ने 50,000 रुपये का माल खरीदा और 5% छू...   47500    Hindi   

                   type  
0       GST Calculation  
1  Interest Calculation  
2       GST Calculation  
3  Interest Calculation  
4  Discount Calculation  


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 640
  Batch size = 16
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile 

 ··········


...


--- Tamil ---
Accuracy: 0.0
F1-score: 0.0
Confusion Matrix:
 [[0 0]
 [1 0]]


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datet

Filter:   0%|          | 0/4 [00:00<?, ? examples/s]


--- English ---
Accuracy: 0.0
F1-score: 0.0
Confusion Matrix:
 [[0 0 0 0]
 [1 0 0 0]
 [0 0 0 0]
 [0 0 1 0]]


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datet

testing 2

In [5]:
# ============================
# 1. Install Required Libraries
# ============================
!pip install -q transformers datasets setfit torch accelerate scikit-learn pandas

# ============================
# 2. Load Dataset (CSV/Excel)
# ============================
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load your dataset
df = pd.read_csv("Financial_dataset.csv")   # or pd.read_excel("financial_dataset.xlsx")

# Expected columns: question, answer, language, type
print("Sample rows:\n", df.head())

# Convert 'type' into categorical labels (classification target)
le = LabelEncoder()
df["label"] = le.fit_transform(df["type"])  # e.g. GST=0, Interest=1, Discount=2

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df[["question", "label", "language", "type"]])

# Split into train/test
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# ============================
# 3. Few-Shot Training with SetFit v2
# ============================
from setfit import SetFitModel, Trainer, TrainingArguments

# Load multilingual backbone
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Training arguments
args = TrainingArguments(
    output_dir="checkpoints",
    batch_size=(16, 2),
    num_epochs=(1, 16),
    num_iterations=20,
    body_learning_rate=2e-5,
    head_learning_rate=1e-2,
    use_amp=False
)

# Trainer with column mapping
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=args,
    metric="accuracy",
    column_mapping={"question": "text", "label": "label"}
)

# Train
trainer.train()

# ============================
# 4. Global Evaluation
# ============================
y_true = test_dataset["label"]
y_pred = trainer.model(test_dataset["question"])

print("Global Accuracy:", accuracy_score(y_true, y_pred))
print("Global F1-score:", f1_score(y_true, y_pred, average="weighted"))
print("Global Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ============================
# 5. Language-Wise Evaluation
# ============================
languages = df["language"].unique()

for lang in languages:
    subset = test_dataset.filter(lambda x: x["language"] == lang)
    if len(subset) == 0:
        continue

    y_true_lang = subset["label"]
    y_pred_lang = trainer.model(subset["question"])

    print(f"\n--- {lang} ---")
    print("Accuracy:", accuracy_score(y_true_lang, y_pred_lang))
    print("F1-score:", f1_score(y_true_lang, y_pred_lang, average="weighted"))
    print("Confusion Matrix:\n", confusion_matrix(y_true_lang, y_pred_lang))

Sample rows:
                                             question  answer language  \
0  एक व्यापारी ने ₹5000 का सामान खरीदा और 18% GST...    5900    Hindi   
1  একজন ব্যবসায়ী ১০,০০০ টাকা ঋণ নিলেন ১০% বার্ষি...   11000  Bengali   
2  ஒரு வியாபாரி ₹2000 மதிப்புள்ள பொருட்களை வாங்கி...    2240    Tamil   
3  A person deposited ₹5000 in a bank at 5% annua...    5500  English   
4  एक कंपनी ने 50,000 रुपये का माल खरीदा और 5% छू...   47500    Hindi   

                   type  
0       GST Calculation  
1  Interest Calculation  
2       GST Calculation  
3  Interest Calculation  
4  Discount Calculation  


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=ut

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 640
  Batch size = 16
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,0.122900


...


--- Tamil ---
Accuracy: 1.0
F1-score: 1.0
Confusion Matrix:
 [[1]]


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datet

Filter:   0%|          | 0/4 [00:00<?, ? examples/s]


--- English ---
Accuracy: 1.0
F1-score: 1.0
Confusion Matrix:
 [[2]]


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datet